In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv('cs-training.csv')
conn = sqlite3.connect('cfr_project.db')
df.to_sql('credit', conn, if_exists='replace', index=False)
print("table loaded")

table loaded


In [2]:
pd.read_sql_query("SELECT COUNT(*) FROM credit", conn)

,COUNT(*)
0,150000


### Q1 — Default rate

In [7]:
query = """
SELECT SeriousDlqin2yrs,
       COUNT(*) AS total,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM credit), 2) AS percentage
FROM credit
GROUP BY SeriousDlqin2yrs;
"""

pd.read_sql(query, conn)

,SeriousDlqin2yrs,total,percentage
0,0,139974,93.32
1,1,10026,6.68


## Business Finding 1

* Only 6.68% of customers defaulted, while 93.32% did not.
* This shows that defaults are relatively rare and the dataset is highly imbalanced.
* Accuracy alone may be misleading when evaluating the model.

### Q2 — Average utilization for defaulters vs non-defaulters

In [8]:
query = """
SELECT SeriousDlqin2yrs,
       ROUND(AVG(RevolvingUtilizationOfUnsecuredLines), 3) AS avg_utilization
FROM credit
GROUP BY SeriousDlqin2yrs;
"""

result = pd.read_sql(query, conn)
result

,SeriousDlqin2yrs,avg_utilization
0,0,6.169
1,1,4.367


### Business Finding 2
* avg utilization is higher for non-defaulters (6.169) vs defaulters (4.367)
* this result is likely influenced by extreme outlier values in the data, not actual behavior.
* the variable should be cleaned before using it for modeling.
                                                                                                                                 

In [10]:
query = """
SELECT MAX(RevolvingUtilizationOfUnsecuredLines),
       MIN(RevolvingUtilizationOfUnsecuredLines)
FROM credit;
"""
pd.read_sql(query, conn)

,MAX(RevolvingUtilizationOfUnsecuredLines),MIN(RevolvingUtilizationOfUnsecuredLines)
0,50708.0,0.0


### RevolvingUtilizationOfUnsecuredLines has a max value of 50,708 — clearly a data entry error.
* Valid range is 0 to 1. All values above 1 will be capped in cleaning phase.
* This outlier was distorting the above results.

### Q3 — Default rate by age group

In [11]:
query = """
SELECT  
CASE 
  WHEN age < 30 THEN 'Under 30'
  WHEN age BETWEEN 30 AND 50 THEN '30 to 50'
  WHEN age BETWEEN 51 AND 65 THEN '51 to 65'
  ELSE 'Above 65'
END AS age_group,
ROUND(AVG(SeriousDlqin2yrs) * 100, 2) AS default_rate_pct
FROM credit
GROUP BY age_group;
"""

pd.read_sql(query, conn)

,age_group,default_rate_pct
0,30 to 50,8.96
1,51 to 65,5.48
2,Above 65,2.41
3,Under 30,11.73


## Business Finding 3

* Default rates decrease with age: under 30 (11.73%), 30–50 (8.96%), 51–65 (5.48%), and above 65 (2.41%).
* Younger customers show the highest risk of default, while older customers are the most reliable.
* Age appears to be a strong predictor of credit risk in this dataset.


### Q4 — Missing values check

In [13]:
query = """
SELECT  
SUM(MonthlyIncome IS NULL) AS missing_income,
SUM(NumberOfDependents IS NULL) AS missing_dependents
FROM credit;
"""

pd.read_sql(query, conn)

,missing_income,missing_dependents
0,29731,3924


## Business Finding 4

* Monthly income has a large number of missing values (29,731), while number of dependents also has missing entries (3,924).
* Income data is heavily incomplete, which may impact model reliability if not handled properly.

### Q5 — Average income of defaulters vs non-defaulters

In [14]:
query = """
SELECT SeriousDlqin2yrs,
       ROUND(AVG(MonthlyIncome), 2) AS avg_income
FROM credit
GROUP BY SeriousDlqin2yrs;
"""

pd.read_sql(query, conn)

,SeriousDlqin2yrs,avg_income
0,0,6747.84
1,1,5630.83


## Business Finding 5

* Non-defaulters have a higher average monthly income (6747.84) compared to defaulters (5630.83).
* This suggests that lower income is associated with higher default risk in this dataset.
* Income appears to be an important factor for predicting credit default.

### Q6 — Most serious late payment analysis

In [15]:
query = """
SELECT NumberOfTimes90DaysLate, 
       COUNT(*) AS total_customers,
       ROUND(AVG(SeriousDlqin2yrs) * 100, 2) AS default_rate_pct
FROM credit
GROUP BY NumberOfTimes90DaysLate
ORDER BY NumberOfTimes90DaysLate;
"""

pd.read_sql(query, conn)

,NumberOfTimes90DaysLate,total_customers,default_rate_pct
0,0,141662,4.63
1,1,5243,33.66
2,2,1555,49.90
3,3,667,57.72
4,4,291,67.01
5,5,131,63.36
6,6,80,60.00
7,7,38,81.58
8,8,21,71.43
9,9,19,73.68


## Business Finding 6

* Customers with 0 late payments have a very low default rate (4.63%), but the risk increases sharply once late payments start appearing.
* Default rate generally rises with the number of 90+ days late payments, reaching very high levels (50%–80%) for repeat defaulters.
* This shows that past severe delinquency is one of the strongest indicators of future credit default risk.